# Intruction
Following [`preprocessing.ipynb`](/notebooks/preprocessing.ipynb), we can now begin with testing our
datasets on different models.

Similarly to our *exploratory data analysis* portion, we will focus on a curated set of agencies to
base our initial model selection. We will focus on the **NYPD**, **DOT**, and **TLC**.

We are again, attempting to determine **resolution time**. This is a regression task. As such, we
will evaluate our models based on **Mean Average Error**, during *EDA* we discovered that there do 
exist many outliers for all three agencies. *MAE* gives us a more robust statistic, and is less
sensitive to outliers than other statistics, like *RMSE*.

# Setup
Here, we will establish our three agencies, as well as create our `X` and `y` features. Since each 
agency contains different magnitudes and fields, we will implement our previously created 
preprocessing pipeline *per-agency*.

In [69]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# grab data
df = pd.read_csv(
    '../data/nyc311_2025.csv', 
    index_col='id', 
    dtype={'zipcode': 'category'},
    parse_dates=['created', 'closed']
)

In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 3475290 entries, 67351762 to 63577994
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   created          datetime64[us]
 1   closed           datetime64[us]
 2   agency_name      str           
 3   problem          str           
 4   detail           str           
 5   borough          str           
 6   lat              float64       
 7   long             float64       
 8   method           str           
 9   zipcode          category      
 10  resolution_time  float64       
dtypes: category(1), datetime64[us](2), float64(3), str(5)
memory usage: 298.3 MB


In [71]:
# initialize agencies

nypd = df[df.agency_name == 'New York City Police Department']
dot = df[df.agency_name == 'Department of Transportation']
tlc = df[df.agency_name == 'Taxi and Limousine Commission']

pd.concat([nypd.head(2), dot.head(2), tlc.head(2)])

,created,closed,agency_name,problem,detail,borough,lat,long,method,zipcode,resolution_time
id,,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,10029,0.684
67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.825137,-73.949447,ONLINE,10031,1.072
67344470,2025-12-31 23:57:00,2026-01-09 01:53:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.871462,-73.830537,UNKNOWN,10475,193.933
67348068,2025-12-31 23:57:00,2026-01-07 09:09:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.861213,-73.825111,UNKNOWN,10475,153.200
67345192,2025-12-31 23:58:25,2026-01-02 11:55:46,Taxi and Limousine Commission,Lost Property,Bag/Wallet,MANHATTAN,40.738077,-73.992123,PHONE,10011,35.956
67348740,2025-12-31 23:20:23,2026-01-02 13:27:51,Taxi and Limousine Commission,Lost Property,Bag/Wallet,QUEENS,40.648320,-73.788281,ONLINE,11430,38.124


In [72]:
agencies = {
    'nypd': {
        'df': nypd
    },
    'dot': {
        'df': dot
    },
    'tlc': {
        'df': tlc
    }
}

## Pipeline import

In [73]:
import cloudpickle as cp

preprocessing_pipeline = None
with open('../models/preprocessing_pipeline.pkl', 'rb') as f:
    preprocessing_pipeline = cp.load(f)

## Train-test split

In [74]:
from sklearn.model_selection import train_test_split

def populate_train_test(agencies: dict):
    '''
    Will give each agency in the agency dict proper X_train, X_test, y_train, y_test splits usign
    `sklearn.model_selection.train_test_split()`
    '''
    
    X = ['created', 'closed', 'detail', 'borough', 'method', 'problem', 'zipcode'] # list of all used features
    y = 'resolution_time'
    for a in agencies.keys():
        curr_df = agencies[a]['df']
        X_train, X_test, y_train, y_test = train_test_split(curr_df[X], curr_df[y], random_state=42, train_size=0.8)
        # populate
        agencies[a]['X_train'] = X_train
        agencies[a]['X_test'] = X_test
        agencies[a]['y_train'] = y_train
        agencies[a]['y_test'] = y_test

In [75]:
populate_train_test(agencies)

In [76]:
for k in agencies.keys():
    print(k, len(agencies[k]['X_train']), len(agencies[k]['X_test']))

nypd 1361864 340466
dot 138912 34729
tlc 22430 5608


# Model Selection
Next, we will begin model selection.

Our process begins with simply creating a pipeline which contains both our `preprocessing_pipeline`
along with the model of interest. 

As stated previouly, we are going to evaluate our models using
**MAE**. Along with using *cross-validation*, with `cv=5`.

We are going to evaluate the following models:
- asdf
-  adsf
- asdf

In [77]:
import warnings
# Our pipeline will create an infrequent column if there are rare problems unseen during pipeline fit
# This is intended behavior. We will ignore warning messages to avoid clutter.
warnings.filterwarnings('ignore', message='Found unknown categories.*', category=UserWarning)

In [ ]:
# model testing function 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

def test_model(agencies: dict, model: any, model_name: str, verbose: int=0):
    for a in agencies.keys():
        # -- define pipeline
        pipe = Pipeline([
            ('preprocessing', preprocessing_pipeline),
            (model_name, model)
        ])

        if verbose > 0:
              print(f'''performing cross validation
        model: {model_name}
        agency: {a}''')

        # -- cross validate
        results = cross_validate(
                pipe,
                agencies[a]['X_train'],
                agencies[a]['y_train'],
                cv=5,
                scoring='neg_mean_absolute_error',
                n_jobs=4,
                verbose=verbose,
                return_estimator=True
        )

        if verbose > 0: 
                print(f'''cross validation results....
        cv_scores (MAE).......................{results['test_score']}
        mean MAE..............................{np.mean(results['test_score'])}''')

        # -- calculate statistics
        best_model = results['estimator'][np.argmax(results['test_score'])] # best model
        y_true = agencies[a]['y_train']
        y_pred = best_model.predict(agencies[a]['X_train'])

        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)

        agencies[a][model_name] = {}
        agencies[a][model_name]['cv_scores'] = results['test_score']
        agencies[a][model_name]['best_r2'] = r2
        agencies[a][model_name]['best_mae'] = mae
        agencies[a][model_name]['best_mape'] = mape

def print_scores(agencies, model_name):
        print(f'All scores for {model_name}:')
        for a in agencies.keys():
                res = agencies[a][model_name]

                print(f'''{a}:
        CV MAE:       {res['cv_scores']}                                                                                                 
                BEST MAE:     {res['best_mae']:.2f}
                BEST MAPE:    {res['best_mape']*100:.2f}
                BEST R^2:     {res['best_r2']:.3f}
''')

In [93]:
from sklearn.linear_model import Ridge

test_model(agencies, Ridge(), 'Ridge')
print_scores(agencies, 'Ridge')

All scores for Ridge:
nypd:
        CV MAE:       [-2.65525778 -2.62711462 -2.6372719  -2.60707535 -2.63774786]                                                                 
                BEST MAE:     2.64
                BEST MAPE:    586.58
                BEST R^2:     0.051

dot:
        CV MAE:       [-250.86116965 -253.22545856 -260.59443126 -262.67508383 -255.03329483]                                                                 
                BEST MAE:     256.71
                BEST MAPE:    5245.32
                BEST R^2:     0.148

tlc:
        CV MAE:       [-1304.58492823 -1286.87281664 -1313.12095549 -1297.27544271
 -1264.26701383]                                                                 
                BEST MAE:     1289.06
                BEST MAPE:    7268.49
                BEST R^2:     0.348



In [94]:
from sklearn.linear_model import ElasticNet

test_model(agencies, ElasticNet(), 'ElasticNet')
print_scores(agencies, 'ElasticNet')

All scores for ElasticNet:
nypd:
        CV MAE:       [-2.71924638 -2.68686383 -2.6960714  -2.66339114 -2.70062885]                                                                 
                BEST MAE:     2.70
                BEST MAPE:    637.26
                BEST R^2:     0.030

dot:
        CV MAE:       [-277.79265806 -283.18567056 -286.86009353 -287.66107597 -283.023716  ]                                                                 
                BEST MAE:     284.41
                BEST MAPE:    11463.47
                BEST R^2:     0.052

tlc:
        CV MAE:       [-1528.7088814  -1502.03518703 -1532.26820386 -1528.55136178
 -1488.93479612]                                                                 
                BEST MAE:     1507.55
                BEST MAPE:    31783.93
                BEST R^2:     0.235



In [95]:
from sklearn.svm import LinearSVR

test_model(agencies, LinearSVR(), 'LinearSVR')
print_scores(agencies, 'LinearSVR')

All scores for LinearSVR:
nypd:
        CV MAE:       [-2.2961234  -2.2618366  -2.27210489 -2.22792809 -2.27620172]                                                                 
                BEST MAE:     2.27
                BEST MAPE:    281.23
                BEST R^2:     0.002

dot:
        CV MAE:       [-187.732743   -193.12569406 -198.84758986 -199.37730521 -193.27381237]                                                                 
                BEST MAE:     194.37
                BEST MAPE:    767.35
                BEST R^2:     0.023

tlc:
        CV MAE:       [-1296.85552685 -1254.03703388 -1295.59716876 -1266.38280489
 -1243.98604554]                                                                 
                BEST MAE:     1268.28
                BEST MAPE:    1147.99
                BEST R^2:     0.206



In [96]:
# xgboost
import xgboost
# .22, .22, .49
test_model(agencies, xgboost.XGBRegressor(), 'XGBoost')
print_scores(agencies, 'XGBoost')

All scores for XGBoost:
nypd:
        CV MAE:       [-2.28164999 -2.2543767  -2.27545373 -2.24823681 -2.26167508]                                                                 
                BEST MAE:     2.25
                BEST MAPE:    414.92
                BEST R^2:     0.220

dot:
        CV MAE:       [-230.5610468  -233.92744118 -243.41723083 -244.41493324 -236.64921003]                                                                 
                BEST MAE:     226.08
                BEST MAPE:    4023.88
                BEST R^2:     0.270

tlc:
        CV MAE:       [-1210.73166259 -1190.65574697 -1219.0779283  -1201.61219016
 -1178.155536  ]                                                                 
                BEST MAE:     1064.83
                BEST MAPE:    5267.14
                BEST R^2:     0.504

